# Análise Estatística Complementar — Interpolação de Precipitação

Este notebook complementa o `interpolation_test.ipynb` com:
1. **Variograma empírico** com modelo exponencial ajustado por mínimos quadrados
2. **Teste de Kruskal-Wallis** — diferenças globais entre os três métodos
3. **Testes pairwise Mann-Whitney U** com correção de Bonferroni
4. **Boxplots com anotações de significância** (prontos para publicação)
5. **Índice de estabilidade** (IQR/mediana) para cada método
6. **Tabela de resumo estatístico** exportável como CSV

> **Pré-requisito:** execute antes as células de leitura de dados e definição das funções `idw`, `kriging`, `splines` e `calcular_metricas` do notebook principal.

## 1. Instalação e Importações

In [ ]:
# Pacotes adicionais necessários (já disponíveis no Colab)
# %pip install scipy seaborn  # descomente se necessário

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.interpolate import griddata
from scipy.spatial.distance import cdist
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("Importações concluídas.")


## 2. Funções Auxiliares

In [ ]:
# ── Cálculo de MAPE ──────────────────────────────────────────────────────────
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = np.abs(y_true) > 1e-6          # evita divisão por zero
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def calcular_metricas(observados, previstos):
    mae  = mean_absolute_error(observados, previstos)
    mse  = mean_squared_error(observados, previstos)
    rmse = np.sqrt(mse)
    mape = calculate_mape(observados, previstos)
    return mae, mse, rmse, mape

# ── Variograma empírico ───────────────────────────────────────────────────────
def variograma_empirico(df, n_bins=10):
    """Calcula o variograma empírico de um DataFrame com colunas
    latitude, longitude e Total."""
    lats = df['latitude'].values
    lons = df['longitude'].values
    vals = df['Total'].values
    n    = len(vals)

    dists, gammas = [], []
    for i in range(n):
        for j in range(i + 1, n):
            d = np.sqrt((lats[i] - lats[j])**2 + (lons[i] - lons[j])**2)
            g = 0.5 * (vals[i] - vals[j])**2
            dists.append(d)
            gammas.append(g)

    dists  = np.array(dists)
    gammas = np.array(gammas)
    limite = np.percentile(dists, 60)          # usa só 60% das distâncias

    bins        = np.linspace(0, limite, n_bins + 1)
    centros, medias, contagens = [], [], []
    for k in range(n_bins):
        mask = (dists >= bins[k]) & (dists < bins[k + 1])
        if mask.sum() >= 2:
            centros.append((bins[k] + bins[k + 1]) / 2)
            medias.append(np.mean(gammas[mask]))
            contagens.append(mask.sum())

    return np.array(centros), np.array(medias), np.array(contagens)

def modelo_exp(h, nugget, sill, range_):
    """Modelo de variograma exponencial."""
    return nugget + sill * (1 - np.exp(-h / range_))

def ajustar_variograma(h, gamma):
    """Ajuste por mínimos quadrados não-lineares (curve_fit)."""
    try:
        popt, _ = curve_fit(
            modelo_exp, h, gamma,
            p0    = [0, np.max(gamma) * 0.9, np.mean(h) * 0.3],
            bounds= ([0, 0, 1e-8], [np.max(gamma)*0.5, np.max(gamma)*2, np.max(h)]),
            maxfev= 8000
        )
    except Exception:
        popt = [0, np.max(gamma), np.mean(h) * 0.3]
    return popt   # (nugget, sill, range)

print("Funções auxiliares definidas.")


## 3. Validação Cruzada Genérica

Estas funções rodam a validação cruzada em **qualquer** dataset que tenha colunas `latitude`, `longitude` e `Total`.
Use-as para SJC, RJ, SP e URU.

In [ ]:
def cross_validate(df, n_iter=2000, split=0.33, seed=42):
    """
    Validação cruzada aleatória (67/33) para IDW, Kriging e Splines.

    Parâmetros
    ----------
    df      : DataFrame com colunas latitude, longitude, Total
    n_iter  : número de divisões aleatórias
    split   : fração destinada ao teste (padrão 0.33)
    seed    : semente para reprodutibilidade

    Retorna
    -------
    dict com listas de (mae, mse, rmse, mape) para cada método
    """
    rng = np.random.RandomState(seed)
    resultados = {'IDW': [], 'Kriging': [], 'Splines': []}
    n = len(df)
    n_test = max(1, int(n * split))

    for _ in range(n_iter):
        test_idx  = rng.choice(n, n_test, replace=False)
        train_idx = np.setdiff1d(np.arange(n), test_idx)
        if len(train_idx) < 3:
            continue

        treino = df.iloc[train_idx].copy()
        teste  = df.iloc[test_idx].copy()

        teste['prev_IDW']    = idw(treino, teste)
        teste['prev_Kriging'] = kriging(treino, teste)
        teste['prev_Splines'] = splines(treino, teste)

        for metodo, col in [('IDW','prev_IDW'),
                             ('Kriging','prev_Kriging'),
                             ('Splines','prev_Splines')]:
            metricas = calcular_metricas(teste['Total'].values,
                                         teste[col].values)
            resultados[metodo].append(metricas)

    # Converte para arrays numpy [n_iter x 4]
    for m in resultados:
        resultados[m] = np.array(resultados[m])

    return resultados


def loocv(df):
    """Leave-One-Out Cross-Validation — ideal para datasets com ≤ 10 estações."""
    resultados = {'IDW': [], 'Kriging': [], 'Splines': []}
    n = len(df)

    for i in range(n):
        treino = df.drop(df.index[i]).copy()
        teste  = df.iloc[[i]].copy()

        teste['prev_IDW']     = idw(treino, teste)
        teste['prev_Kriging'] = kriging(treino, teste)
        teste['prev_Splines'] = splines(treino, teste)

        for metodo, col in [('IDW','prev_IDW'),
                             ('Kriging','prev_Kriging'),
                             ('Splines','prev_Splines')]:
            metricas = calcular_metricas(teste['Total'].values,
                                         teste[col].values)
            resultados[metodo].append(metricas)

    for m in resultados:
        resultados[m] = np.array(resultados[m])

    return resultados

print("Funções de validação cruzada definidas.")


## 4. Testes Estatísticos

### 4.1 Kruskal-Wallis + Mann-Whitney U com correção de Bonferroni

Usamos testes não-paramétricos pois as distribuições de MAPE são tipicamente assimétricas (skewed).

In [ ]:
def testes_estatisticos(resultados, nome_dataset='Dataset', metrica=3):
    """
    Executa Kruskal-Wallis e pairwise Mann-Whitney U (Bonferroni) sobre
    a distribuição de uma métrica de erro específica.

    Parâmetros
    ----------
    resultados   : dict retornado por cross_validate() ou loocv()
    nome_dataset : string para identificar o dataset na impressão
    metrica      : índice da métrica — 0=MAE, 1=MSE, 2=RMSE, 3=MAPE

    Retorna
    -------
    dict com resultados dos testes
    """
    nomes_metricas = ['MAE', 'MSE', 'RMSE', 'MAPE']
    nome_metrica   = nomes_metricas[metrica]
    metodos        = list(resultados.keys())

    # Extrai os valores da métrica escolhida
    arrays = {m: resultados[m][:, metrica] for m in metodos}

    print(f"\n{'='*55}")
    print(f" {nome_dataset}  —  métrica: {nome_metrica}")
    print(f"{'='*55}")

    # Estatísticas descritivas
    for m, arr in arrays.items():
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        print(f"  {m:10s}  mediana={med:7.2f}  IQR=[{q1:.2f}, {q3:.2f}]")

    # Kruskal-Wallis
    kw_stat, kw_p = stats.kruskal(*[arrays[m] for m in metodos])
    print(f"\n  Kruskal-Wallis:  H = {kw_stat:.3f},  p = {kw_p:.4e}")

    pares    = [('IDW','Kriging'), ('IDW','Splines'), ('Kriging','Splines')]
    k_comp   = len(pares)         # fator de correção Bonferroni
    resultados_pares = {}

    if kw_p < 0.05:
        print(f"  → diferenças significativas detectadas; teste pairwise (Bonferroni k={k_comp}):")
        for (m1, m2) in pares:
            if m1 not in arrays or m2 not in arrays:
                continue
            _, p_raw    = stats.mannwhitneyu(arrays[m1], arrays[m2],
                                             alternative='two-sided')
            p_adj       = min(p_raw * k_comp, 1.0)
            sig         = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01
                          else ('*' if p_adj < 0.05 else 'ns'))
            resultados_pares[f"{m1}_vs_{m2}"] = (p_adj, sig)
            print(f"    {m1:10s} vs {m2:10s}:  p_adj = {p_adj:.4e}  {sig}")
    else:
        print("  → diferenças NÃO significativas (p ≥ 0.05). Testes pairwise omitidos.")
        for (m1, m2) in pares:
            resultados_pares[f"{m1}_vs_{m2}"] = (1.0, 'ns')

    return {'KW': (kw_stat, kw_p), 'pares': resultados_pares,
            'arrays': arrays, 'metrica': nome_metrica}

print("Função de testes estatísticos definida.")


## 5. Execução para o Dataset SJC

Ajuste `novas_loc` para o seu DataFrame, ou substitua pelo dataset de interesse.

In [ ]:
# ── Rode esta célula após carregar novas_loc ──────────────────────────────────
# Validação cruzada aleatória (2000 iterações, split 67/33)
print("Executando validação cruzada SJC (2000 iterações)...")
res_sjc = cross_validate(novas_loc, n_iter=2000, split=0.33, seed=42)
print("Concluído.")

# Testes estatísticos sobre o MAPE (índice 3)
stat_sjc = testes_estatisticos(res_sjc, nome_dataset='SJC', metrica=3)


## 6. Boxplot com Anotações de Significância Estatística

In [ ]:
def boxplot_significancia(resultados_dict, stat_dict, titulo='Dataset',
                          metrica=3, salvar=None):
    """
    Gera boxplot de MAPE (ou outra métrica) com colchetes de significância.

    Parâmetros
    ----------
    resultados_dict : dict retornado por cross_validate()
    stat_dict       : dict retornado por testes_estatisticos()
    titulo          : nome do dataset (usado no título e no nome do arquivo)
    metrica         : índice — 0=MAE, 1=MSE, 2=RMSE, 3=MAPE
    salvar          : caminho para salvar a figura (None = só exibir)
    """
    nomes_metricas = ['MAE', 'MSE', 'RMSE', 'MAPE']
    nome_metrica   = nomes_metricas[metrica]

    metodos = list(resultados_dict.keys())
    cores   = {'IDW': '#378ADD', 'Kriging': '#1D9E75', 'Splines': '#D85A30'}
    dados   = [resultados_dict[m][:, metrica] for m in metodos]

    fig, ax = plt.subplots(figsize=(7, 5))

    bp = ax.boxplot(
        dados, patch_artist=True, widths=0.5,
        medianprops  = dict(color='black', linewidth=2),
        flierprops   = dict(marker='o', markersize=2, alpha=0.25),
        whiskerprops = dict(linewidth=1.2),
        capprops     = dict(linewidth=1.2),
        boxprops     = dict(linewidth=1.2),
    )
    for patch, m in zip(bp['boxes'], metodos):
        patch.set_facecolor(cores[m])
        patch.set_alpha(0.75)

    # Anotações de significância
    pares_plot = [('IDW','Kriging',1,2),
                  ('IDW','Splines',1,3),
                  ('Kriging','Splines',2,3)]
    y_base = max(np.percentile(d, 95) for d in dados)
    passo  = y_base * 0.10
    h_barra= passo * 0.25

    for (m1, m2, x1, x2) in pares_plot:
        chave = f"{m1}_vs_{m2}"
        if chave not in stat_dict['pares']:
            continue
        p_adj, sig = stat_dict['pares'][chave]
        if sig == 'ns':
            continue
        y = y_base + passo
        ax.plot([x1, x1, x2, x2], [y, y+h_barra, y+h_barra, y],
                lw=1, c='#444')
        ax.text((x1+x2)/2, y+h_barra, sig,
                ha='center', va='bottom', fontsize=10, color='#222')
        y_base = y + h_barra + passo * 0.3

    # Valor p do KW no canto
    kw_p = stat_dict['KW'][1]
    kw_txt = f'KW p{"<0.001" if kw_p < 0.001 else f"={kw_p:.3f}"}'
    ax.text(0.97, 0.97, kw_txt, transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='#555', style='italic')

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(metodos, fontsize=11)
    ax.set_ylabel(f'{nome_metrica}', fontsize=11)
    ax.set_title(f'{titulo} — distribuição de {nome_metrica}\n'
                 f'(* p<0.05  ** p<0.01  *** p<0.001, correção Bonferroni)',
                 fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    patches = [mpatches.Patch(facecolor=cores[m], alpha=0.75, label=m)
               for m in metodos]
    ax.legend(handles=patches, frameon=False, fontsize=10)

    plt.tight_layout()
    if salvar:
        plt.savefig(salvar, dpi=200, bbox_inches='tight')
        print(f'Figura salva em: {salvar}')
    plt.show()


# ── Gera o boxplot para SJC ───────────────────────────────────────────────────
boxplot_significancia(res_sjc, stat_sjc,
                      titulo='SJC (n=28)',
                      metrica=3,
                      salvar='boxplot_sjc_significancia.png')


### 6.1 Boxplot com todas as métricas (MAE, MSE, RMSE, MAPE)

In [ ]:
def boxplot_todas_metricas(resultados_dict, titulo='Dataset', salvar=None):
    """Gera um painel 2×2 com boxplots para as quatro métricas de erro."""
    nomes = ['MAE', 'MSE', 'RMSE', 'MAPE']
    cores = {'IDW': '#378ADD', 'Kriging': '#1D9E75', 'Splines': '#D85A30'}
    metodos = list(resultados_dict.keys())

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()

    for idx, (nome_m, ax) in enumerate(zip(nomes, axes)):
        dados = [resultados_dict[m][:, idx] for m in metodos]
        bp = ax.boxplot(
            dados, patch_artist=True, widths=0.5,
            medianprops  = dict(color='black', linewidth=2),
            flierprops   = dict(marker='o', markersize=2, alpha=0.2),
            whiskerprops = dict(linewidth=1.2),
            capprops     = dict(linewidth=1.2),
        )
        for patch, m in zip(bp['boxes'], metodos):
            patch.set_facecolor(cores[m])
            patch.set_alpha(0.75)
        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels(metodos, fontsize=10)
        ax.set_ylabel(nome_m, fontsize=10)
        ax.set_title(nome_m, fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    patches = [mpatches.Patch(facecolor=cores[m], alpha=0.75, label=m)
               for m in metodos]
    fig.legend(handles=patches, loc='lower center', ncol=3,
               fontsize=11, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f'{titulo} — Distribuição das métricas de erro\n'
                 f'(validação cruzada, {len(resultados_dict[metodos[0]])} iterações)',
                 fontsize=12)
    plt.tight_layout()
    if salvar:
        plt.savefig(salvar, dpi=200, bbox_inches='tight')
        print(f'Figura salva em: {salvar}')
    plt.show()


boxplot_todas_metricas(res_sjc, titulo='SJC (n=28)',
                       salvar='boxplot_sjc_todas_metricas.png')


## 7. Variograma Empírico

O variograma caracteriza a estrutura de correlação espacial da precipitação e explica mecanisticamente por que cada método performa melhor ou pior em cada dataset.

In [ ]:
def plotar_variograma(df, nome_dataset='Dataset', cor='#185FA5',
                      n_bins=10, salvar=None):
    """
    Plota o variograma empírico e o modelo exponencial ajustado.

    Parâmetros
    ----------
    df           : DataFrame com colunas latitude, longitude, Total
    nome_dataset : nome do dataset para o título
    cor          : cor dos pontos empiricos
    n_bins       : número de classes de distância
    salvar       : caminho para salvar (None = só exibir)

    Retorna
    -------
    (nugget, sill, range_ajustado)
    """
    h_bins, gamma_bins, contagens = variograma_empirico(df, n_bins=n_bins)
    nugget, sill, range_ = ajustar_variograma(h_bins, gamma_bins)

    h_smooth     = np.linspace(0, h_bins.max() * 1.1, 300)
    gamma_fitted = modelo_exp(h_smooth, nugget, sill, range_)

    fig, ax = plt.subplots(figsize=(7, 4.5))

    # Tamanho do marcador proporcional ao número de pares
    tamanhos = 30 + contagens * 2
    ax.scatter(h_bins, gamma_bins, s=tamanhos, color=cor,
               zorder=5, label='Variograma empírico', alpha=0.85)
    ax.plot(h_smooth, gamma_fitted, color='#333', linewidth=2,
            label=f'Modelo ajustado\n'
                  f'$C_0$={nugget:.1f}  $C$={sill:.1f}  $a$={range_:.4f}°',
            zorder=4)

    # Linhas de referência
    ax.axhline(nugget + sill, color='gray', linestyle=':', linewidth=1,
               alpha=0.7, label=f'Patamar (sill = {nugget+sill:.1f})')
    ax.axvline(range_, color='steelblue', linestyle='--',
               linewidth=1, alpha=0.7)
    ax.text(range_ + h_bins.max()*0.02, gamma_bins.min()*0.5,
            f'range={range_:.4f}°', fontsize=9, color='steelblue')

    ax.set_xlabel('Distância de lag (°)', fontsize=11)
    ax.set_ylabel('Semivariância (mm²)', fontsize=11)
    ax.set_title(f'Variograma empírico — {nome_dataset}\n'
                 f'(tamanho do marcador ∝ nº de pares de estações)',
                 fontsize=11)
    ax.legend(fontsize=9, frameon=False)
    ax.grid(alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if salvar:
        plt.savefig(salvar, dpi=200, bbox_inches='tight')
        print(f'Figura salva em: {salvar}')
    plt.show()

    print(f"\nParâmetros ajustados — {nome_dataset}:")
    print(f"  Nugget (C0) = {nugget:.3f}")
    print(f"  Sill   (C)  = {sill:.3f}")
    print(f"  Range  (a)  = {range_:.6f}° ≈ {range_*111:.1f} km")

    return nugget, sill, range_


# ── Plota o variograma para SJC ──────────────────────────────────────────────
params_sjc = plotar_variograma(novas_loc, nome_dataset='SJC (n=28)',
                                cor='#7F77DD',
                                salvar='variograma_sjc.png')


## 8. Índice de Estabilidade

**Definição:** `IQR / mediana` — quanto menor, mais consistente é o método independentemente da configuração espacial das estações.

In [ ]:
def indice_estabilidade(resultados_dict, nome_dataset='Dataset',
                         metrica=3, salvar=None):
    """
    Exibe e plota o índice de estabilidade (IQR/mediana) para cada método.
    """
    nomes_metricas = ['MAE', 'MSE', 'RMSE', 'MAPE']
    nome_metrica   = nomes_metricas[metrica]
    cores          = {'IDW': '#378ADD', 'Kriging': '#1D9E75', 'Splines': '#D85A30'}
    metodos        = list(resultados_dict.keys())

    indices, medianas = [], []
    print(f"\nÍndice de estabilidade ({nome_metrica}) — {nome_dataset}")
    print(f"  {'Método':12s}  {'Mediana':>10s}  {'IQR':>10s}  {'IQR/Mediana':>12s}")
    for m in metodos:
        arr = resultados_dict[m][:, metrica]
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        iqr  = q3 - q1
        idx  = iqr / med if med > 0 else 0
        print(f"  {m:12s}  {med:10.3f}  {iqr:10.3f}  {idx:12.4f}")
        indices.append(idx)
        medianas.append(med)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Mediana MAPE
    bars = axes[0].bar(metodos, medianas,
                       color=[cores[m] for m in metodos], alpha=0.78,
                       edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, medianas):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + max(medianas)*0.01,
                     f'{v:.2f}', ha='center', va='bottom', fontsize=10)
    axes[0].set_ylabel(f'Mediana {nome_metrica}', fontsize=11)
    axes[0].set_title('Mediana do erro', fontsize=11, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    # Índice de estabilidade
    bars2 = axes[1].bar(metodos, indices,
                        color=[cores[m] for m in metodos], alpha=0.78,
                        edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars2, indices):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + max(indices)*0.01,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    axes[1].set_ylabel('IQR / Mediana', fontsize=11)
    axes[1].set_title('Índice de estabilidade\n(menor = mais consistente)',
                      fontsize=11, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

    fig.suptitle(f'{nome_dataset} — desempenho e estabilidade ({nome_metrica})',
                 fontsize=12)
    plt.tight_layout()
    if salvar:
        plt.savefig(salvar, dpi=200, bbox_inches='tight')
        print(f'Figura salva em: {salvar}')
    plt.show()


indice_estabilidade(res_sjc, nome_dataset='SJC (n=28)', metrica=3,
                    salvar='estabilidade_sjc.png')


## 9. Tabela de Resumo Estatístico

Gera uma tabela formatada com mediana, IQR e resultados dos testes — pronta para exportar como CSV ou copiar para o LaTeX.

In [ ]:
def tabela_resumo(resultados_datasets, stat_datasets, metrica=3):
    """
    Consolida os resultados de múltiplos datasets em uma única tabela.

    Parâmetros
    ----------
    resultados_datasets : dict  {nome_dataset: dict retornado por cross_validate()}
    stat_datasets       : dict  {nome_dataset: dict retornado por testes_estatisticos()}
    metrica             : índice da métrica (padrão 3 = MAPE)
    """
    nomes_metricas = ['MAE', 'MSE', 'RMSE', 'MAPE']
    nome_metrica   = nomes_metricas[metrica]
    metodos        = ['IDW', 'Kriging', 'Splines']
    linhas         = []

    for nome_ds, res in resultados_datasets.items():
        stat = stat_datasets[nome_ds]
        for m in metodos:
            arr       = res[m][:, metrica]
            q1, med, q3 = np.percentile(arr, [25, 50, 75])
            kw_p      = stat['KW'][1]
            pares_sig = []
            for par, (p_adj, sig) in stat['pares'].items():
                if m in par and sig != 'ns':
                    outro = par.replace(m,'').replace('_vs_','').strip('_')
                    pares_sig.append(f"{outro}({sig})")
            linhas.append({
                'Dataset'    : nome_ds,
                'Método'     : m,
                f'Mediana {nome_metrica}': round(med, 3),
                'Q1'         : round(q1, 3),
                'Q3'         : round(q3, 3),
                'IQR'        : round(q3 - q1, 3),
                'IQR/Mediana': round((q3-q1)/med if med > 0 else 0, 4),
                'KW p-valor' : f"{kw_p:.4e}",
                'Sig. vs'    : ', '.join(pares_sig) if pares_sig else '—',
            })

    tabela = pd.DataFrame(linhas)
    print(tabela.to_string(index=False))
    return tabela


# ── Exemplo de uso: passe os dicts de todos os datasets ──────────────────────
# Após rodar cross_validate() e testes_estatisticos() para cada dataset, use:
#
# tabela = tabela_resumo(
#     resultados_datasets = {'SJC': res_sjc, 'RJ': res_rj, 'SP': res_sp, 'URU': res_uru},
#     stat_datasets       = {'SJC': stat_sjc, 'RJ': stat_rj, 'SP': stat_sp, 'URU': stat_uru},
# )
# tabela.to_csv('resumo_estatistico.csv', index=False)
#
# Para um único dataset (SJC):
tabela = tabela_resumo({'SJC': res_sjc}, {'SJC': stat_sjc})
tabela.to_csv('resumo_estatistico_sjc.csv', index=False)
print("\nTabela exportada para resumo_estatistico_sjc.csv")


## 10. Template para Outros Datasets (RJ, SP, URU)

Substitua `df_rj`, `df_sp`, `df_uru` pelos seus DataFrames reais com colunas `latitude`, `longitude`, `Total`.

In [ ]:
# ── TEMPLATE — copie e adapte para cada dataset ───────────────────────────────
#
# Para datasets com poucas estações (RJ = 5, SP = 10), prefira LOOCV:
#   res_rj  = loocv(df_rj)
#   stat_rj = testes_estatisticos(res_rj, nome_dataset='RJ', metrica=3)
#
# Para URU (16 estações) use validação cruzada aleatória:
#   res_uru  = cross_validate(df_uru, n_iter=2000, split=0.33, seed=42)
#   stat_uru = testes_estatisticos(res_uru, nome_dataset='URU', metrica=3)
#
# Gere as figuras para cada um:
#   boxplot_significancia(res_rj,  stat_rj,  titulo='RJ (n=5)',   salvar='boxplot_rj.png')
#   plotar_variograma(df_rj,  nome_dataset='RJ',  cor='#185FA5',  salvar='variograma_rj.png')
#   indice_estabilidade(res_rj, nome_dataset='RJ', salvar='estabilidade_rj.png')
#
# E ao final consolide tudo:
#   tabela_final = tabela_resumo(
#       {'SJC': res_sjc, 'RJ': res_rj, 'SP': res_sp, 'URU': res_uru},
#       {'SJC': stat_sjc, 'RJ': stat_rj, 'SP': stat_sp, 'URU': stat_uru},
#   )
#   tabela_final.to_csv('resumo_completo.csv', index=False)

print("Template pronto — adapte df_rj / df_sp / df_uru e execute.")


## 11. Gráfico de Barras — Mediana MAPE (LOOCV)

Replica e melhora o gráfico original `SJCpap.png` do notebook principal.

In [ ]:
def barplot_loocv(resultados_dict, nome_dataset='Dataset',
                  metrica=3, salvar=None):
    """Gráfico de barras da mediana MAPE no estilo do notebook original,
    com barras de erro mostrando o IQR."""
    nomes_metricas = ['MAE', 'MSE', 'RMSE', 'MAPE']
    nome_metrica   = nomes_metricas[metrica]
    cores   = {'IDW': '#378ADD', 'Kriging': '#1D9E75', 'Splines': '#D85A30'}
    metodos = list(resultados_dict.keys())

    medianas = [np.median(resultados_dict[m][:, metrica]) for m in metodos]
    q1s      = [np.percentile(resultados_dict[m][:, metrica], 25) for m in metodos]
    q3s      = [np.percentile(resultados_dict[m][:, metrica], 75) for m in metodos]
    erros    = [[m - q1 for m, q1 in zip(medianas, q1s)],
                [q3 - m for m, q3 in zip(medianas, q3s)]]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(metodos, medianas,
                  color=[cores[m] for m in metodos],
                  alpha=0.78, edgecolor='white', linewidth=0.8,
                  yerr=erros, capsize=6,
                  error_kw=dict(elinewidth=1.5, ecolor='#444', capthick=1.5))

    for bar, val in zip(bars, medianas):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(medianas)*0.02,
                f'{val:.1f}%', ha='center', va='bottom',
                fontsize=11, fontweight='bold')

    ax.set_ylabel(f'Mediana {nome_metrica} (%)', fontsize=11)
    ax.set_title(f'{nome_dataset} — Mediana {nome_metrica} por Método\n'
                 f'(barras de erro = IQR [Q1, Q3])', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    if salvar:
        plt.savefig(salvar, dpi=200, bbox_inches='tight')
        print(f'Figura salva em: {salvar}')
    plt.show()


barplot_loocv(res_sjc, nome_dataset='SJC (n=28)',
              salvar='barplot_mediana_mape_sjc.png')
